# 10b — Questionnaire Baseline Models

This notebook evaluates baseline classifiers using the cleaned questionnaire dataset produced by `02_questionnaire_cleaning.ipynb`.

## Objective

The notebook:

- loads `data/processed/questionnaire_cleaned.csv`;
- selects only questionnaire-derived predictors;
- creates reproducible stratified train, validation, and test subsets;
- uses the shared preprocessing and modeling framework in `src/modeling`;
- evaluates the Dummy Classifier, Logistic Regression, and Random Forest;
- uses the project-configured class weighting where applicable;
- reports validation and test metrics;
- reports per-class precision and recall;
- produces confusion matrices;
- saves questionnaire-specific outputs using `src/modeling/outputs.py`;
- provides initial observations from the questionnaire model results.



### Libraries and project paths

In [24]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

QUESTIONNAIRE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "questionnaire_cleaned.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Questionnaire file: {QUESTIONNAIRE_FILE}")

Project root: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Questionnaire file: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\questionnaire_cleaned.csv


### 2. Importing the Shared Modeling Framework

In [25]:
from src.modeling.preprocessing import (
    prepare_dataset,
    identify_feature_types,
)

from src.modeling.baseline import get_dummy_classifier

from src.modeling.models import (
    get_logistic_regression,
    get_random_forest,
)

from src.modeling.workflow import run_models

from src.modeling.outputs import (
    save_metrics,
    save_classification_report,
    save_confusion_matrix,
    save_model_comparison,
    save_table,
    save_confusion_matrix_figure,
)

from src.modeling.config import (
    TARGET_COLUMN,
    TRAIN_SIZE,
    VALIDATION_SIZE,
    TEST_SIZE,
    RANDOM_STATE,
    CLASS_WEIGHT,
    USE_CLASS_WEIGHT,
    PRIMARY_METRIC,
)

### Loading questionnaire data 

In [26]:
assert QUESTIONNAIRE_FILE.exists(), (
    f"Clean questionnaire dataset not found: {QUESTIONNAIRE_FILE}\n"
    "Run 02_questionnaire_cleaning.ipynb first."
)

questionnaire = pd.read_csv(
    QUESTIONNAIRE_FILE,
    dtype={"patient_id": str},
)

print(f"Dataset shape: {questionnaire.shape}")
print(
    f"Unique participants: "
    f"{questionnaire['patient_id'].nunique():,}"
)

display(questionnaire.head())

Dataset shape: (469, 46)
Unique participants: 469


,patient_id,questionnaire_name,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30,questions_answered,questions_missing,questionnaire_status,total_symptom_count,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30,0,Complete,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,NMS,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,30,0,Complete,12.0,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,3.0
2,3,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30,0,Complete,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,NMS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,30,0,Complete,12.0,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,2.0,3.0
4,5,NMS,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,30,0,Complete,11.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,5.0


### Inspecting questionnaire features

In [27]:
print("Questionnaire columns:")

for column in questionnaire.columns:
    print(column)

Questionnaire columns:
patient_id
questionnaire_name
Q01
Q02
Q03
Q04
Q05
Q06
Q07
Q08
Q09
Q10
Q11
Q12
Q13
Q14
Q15
Q16
Q17
Q18
Q19
Q20
Q21
Q22
Q23
Q24
Q25
Q26
Q27
Q28
Q29
Q30
questions_answered
questions_missing
questionnaire_status
total_symptom_count
gastrointestinal_count
urinal_count
pain_count
miscellaneous_count
apathy_attention_memory_count
distortion_perception_count
depression_anxiety_count
sexual_function_count
cardiovascular_count
sleep_fatigue_count


### Identifying questionnaire items 

In [28]:
questionnaire_items = [
    f"Q{i:02d}"
    for i in range(1, 31)
]

existing_questionnaire_items = [
    column
    for column in questionnaire_items
    if column in questionnaire.columns
]

print(
    "Questionnaire items found:",
    len(existing_questionnaire_items)
)

print(existing_questionnaire_items)

Questionnaire items found: 30
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30']


In [29]:
non_feature_columns = {
    "patient_id",
    "questionnaire_name",
    "study_id",
    "label",
    "condition_original",
    "condition_group",

    # Questionnaire quality-control variables
    "questions_answered",
    "questions_missing",
    "questionnaire_status",
}

derived_questionnaire_features = [
    column
    for column in questionnaire.columns
    if column not in non_feature_columns
    and column not in existing_questionnaire_items
]

print("Derived questionnaire features:")

for column in derived_questionnaire_features:
    print(column)

Derived questionnaire features:
total_symptom_count
gastrointestinal_count
urinal_count
pain_count
miscellaneous_count
apathy_attention_memory_count
distortion_perception_count
depression_anxiety_count
sexual_function_count
cardiovascular_count
sleep_fatigue_count


In [30]:
questionnaire_features = (
    existing_questionnaire_items
    + derived_questionnaire_features
)

questionnaire_features = list(
    dict.fromkeys(questionnaire_features)
)

print(
    "Number of questionnaire predictors:",
    len(questionnaire_features)
)

print(questionnaire_features)

Number of questionnaire predictors: 41
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'total_symptom_count', 'gastrointestinal_count', 'urinal_count', 'pain_count', 'miscellaneous_count', 'apathy_attention_memory_count', 'distortion_perception_count', 'depression_anxiety_count', 'sexual_function_count', 'cardiovascular_count', 'sleep_fatigue_count']


In [31]:
forbidden_columns = [
    "patient_id",
    "study_id",
    TARGET_COLUMN,
    "condition_original",
    "condition_group",
    "questionnaire_name",
]

remaining_forbidden = [
    column
    for column in forbidden_columns
    if column in questionnaire_features
]

assert not remaining_forbidden, (
    f"Leakage-related columns included as predictors: "
    f"{remaining_forbidden}"
)

print("Questionnaire leakage check: PASS")

Questionnaire leakage check: PASS


### building the questionnaire modeling table

## 6. Attaching the Diagnostic Target

The cleaned questionnaire file contains questionnaire-derived predictors but does not contain the diagnostic target. The target label is therefore attached using `patient_id` from the integrated participant dataset.

In [32]:
INTEGRATED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "multimodal_full_task_aware.csv"
)

assert INTEGRATED_FILE.exists(), (
    f"Integrated dataset not found: {INTEGRATED_FILE}"
)

integrated = pd.read_csv(
    INTEGRATED_FILE,
    dtype={"patient_id": str},
)

labels = integrated[
    ["patient_id", TARGET_COLUMN]
].copy()

questionnaire_df = questionnaire[
    ["patient_id"] + questionnaire_features
].merge(
    labels,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

print("Modeling table shape:", questionnaire_df.shape)
print(
    "Unique participants:",
    questionnaire_df["patient_id"].nunique()
)
print(
    "Missing target values:",
    questionnaire_df[TARGET_COLUMN].isna().sum()
)

Modeling table shape: (469, 43)
Unique participants: 469
Missing target values: 0


### Checking the questionnaire data

In [33]:
print("Class distribution:")

display(
    questionnaire_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("\nMissing values in predictors:")

missing_values = (
    questionnaire_df[questionnaire_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_values
    .rename("missing")
    .to_frame()
    .head(20)
)

duplicate_ids = (
    questionnaire_df["patient_id"]
    .duplicated()
    .sum()
)

print(
    "\nDuplicate participant IDs:",
    duplicate_ids
)

assert duplicate_ids == 0

Class distribution:


,count
label,
0,79
1,276
2,114



Missing values in predictors:


,missing
Q01,0
Q02,0
Q03,0
Q04,0
Q05,0
Q06,0
Q07,0
Q08,0
Q09,0
Q10,0



Duplicate participant IDs: 0


### Creating Reproducible Stratified Subsets

The questionnaire modality uses the same project-level split proportions and random state as the other modalities to support fair comparison.

In [34]:
remaining_size = (
    VALIDATION_SIZE
    + TEST_SIZE
)

train_questionnaire, remaining_questionnaire = (
    train_test_split(
        questionnaire_df,
        test_size=remaining_size,
        stratify=questionnaire_df[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
)

test_fraction_of_remaining = (
    TEST_SIZE / remaining_size
)

validation_questionnaire, test_questionnaire = (
    train_test_split(
        remaining_questionnaire,
        test_size=test_fraction_of_remaining,
        stratify=remaining_questionnaire[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
)

print(
    f"Train shape:      {train_questionnaire.shape}"
)
print(
    f"Validation shape: {validation_questionnaire.shape}"
)
print(
    f"Test shape:       {test_questionnaire.shape}"
)

print("\nParticipant overlap checks:")

print(
    "Train/validation overlap:",
    bool(
        set(train_questionnaire.patient_id)
        & set(validation_questionnaire.patient_id)
    )
)

print(
    "Train/test overlap:",
    bool(
        set(train_questionnaire.patient_id)
        & set(test_questionnaire.patient_id)
    )
)

print(
    "Validation/test overlap:",
    bool(
        set(validation_questionnaire.patient_id)
        & set(test_questionnaire.patient_id)
    )
)

Train shape:      (328, 43)
Validation shape: (70, 43)
Test shape:       (71, 43)

Participant overlap checks:
Train/validation overlap: False
Train/test overlap: False
Validation/test overlap: False


### Checking class distribution across splits 

In [35]:
split_distribution = pd.DataFrame({
    "Train":
        train_questionnaire[
            TARGET_COLUMN
        ].value_counts().sort_index(),

    "Validation":
        validation_questionnaire[
            TARGET_COLUMN
        ].value_counts().sort_index(),

    "Test":
        test_questionnaire[
            TARGET_COLUMN
        ].value_counts().sort_index(),
}).fillna(0).astype(int)

split_distribution

,Train,Validation,Test
label,,,
0,55,12,12
1,193,41,42
2,80,17,17


### Shared Preprocessing and Leakage Validation

In [36]:
X_train, y_train, preprocessing = prepare_dataset(
    train_questionnaire
)

numerical_features, categorical_features = (
    identify_feature_types(X_train)
)

print(
    "Predictor columns passed to preprocessing:"
)

print(X_train.columns.tolist())

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

excluded_check = [
    "patient_id",
    "study_id",
    TARGET_COLUMN,
    "condition_original",
    "condition_group",
    "questionnaire_name",
]

remaining_excluded = [
    column
    for column in excluded_check
    if column in X_train.columns
]

assert not remaining_excluded, (
    "Leakage/identifier columns remain "
    f"in predictors: {remaining_excluded}"
)

print("\nLeakage prevention check: PASS")

preprocessing

Predictor columns passed to preprocessing:
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'total_symptom_count', 'gastrointestinal_count', 'urinal_count', 'pain_count', 'miscellaneous_count', 'apathy_attention_memory_count', 'distortion_perception_count', 'depression_anxiety_count', 'sexual_function_count', 'cardiovascular_count', 'sleep_fatigue_count']

Numerical features:
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'total_symptom_count', 'gastrointestinal_count', 'urinal_count', 'pain_count', 'miscellaneous_count', 'apathy_attention_memory_count', 'distortion_perception_count', 'depression_anxiety_count', 'sexual_function_count', 'cardiovascular_count', 'slee

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

### Defining Baseline models 

In [37]:
models = {
    "Dummy":
        get_dummy_classifier(),

    "Logistic Regression":
        get_logistic_regression(),

    "Random Forest":
        get_random_forest(),
}

print(
    "Class weighting enabled:",
    USE_CLASS_WEIGHT
)

print(
    "Configured class weight:",
    CLASS_WEIGHT
)

print(
    "Primary metric:",
    PRIMARY_METRIC
)

models

Class weighting enabled: True
Configured class weight: balanced
Primary metric: macro_f1


{'Dummy': DummyClassifier(random_state=42, strategy='most_frequent'),
 'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=5,
                        random_state=42)}

### Running questionnaire Baseline models 

In [38]:
results = run_models(
    models=models,
    train_df=train_questionnaire,
    validation_df=validation_questionnaire,
    test_df=test_questionnaire,
)

print("Completed models:")
print(list(results.keys()))

Completed models:
['Dummy', 'Logistic Regression', 'Random Forest']


### Validating Model Comparison

In [39]:
validation_comparison = pd.DataFrame({
    model_name: result["metrics"]
    for model_name, result in results.items()
}).T

validation_comparison = (
    validation_comparison
    .sort_values(
        PRIMARY_METRIC,
        ascending=False,
    )
)

validation_comparison

,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
Logistic Regression,0.628571,0.642715,0.586445,0.573500,0.642715
Random Forest,0.657143,0.636019,0.573148,0.608211,0.636019
Dummy,0.585714,0.333333,0.246246,0.195238,0.333333


### Testing Model Comparison

In [40]:
test_comparison = pd.DataFrame({
    model_name: result["test_metrics"]
    for model_name, result in results.items()
}).T

test_comparison = (
    test_comparison
    .sort_values(
        PRIMARY_METRIC,
        ascending=False,
    )
)

test_comparison

,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
Logistic Regression,0.619718,0.604809,0.582996,0.576300,0.604809
Random Forest,0.661972,0.598273,0.565196,0.600223,0.598273
Dummy,0.591549,0.333333,0.247788,0.197183,0.333333


### Per-Class Precision and Recall

In [41]:
for model_name, result in results.items():

    print("=" * 70)
    print(
        f"{model_name.upper()} — VALIDATION"
    )
    print("=" * 70)

    display(
        result["classification_report"]
    )

    print(
        f"\n{model_name.upper()} — TEST"
    )

    display(
        result["test_report"]
    )

    print()

DUMMY — VALIDATION


,precision,recall,f1-score,support
0,0.000000,0.000000,0.000000,12.000000
1,0.585714,1.000000,0.738739,41.000000
2,0.000000,0.000000,0.000000,17.000000
accuracy,0.585714,0.585714,0.585714,0.585714
macro avg,0.195238,0.333333,0.246246,70.000000
weighted avg,0.343061,0.585714,0.432690,70.000000



DUMMY — TEST


,precision,recall,f1-score,support
0,0.000000,0.000000,0.000000,12.000000
1,0.591549,1.000000,0.743363,42.000000
2,0.000000,0.000000,0.000000,17.000000
accuracy,0.591549,0.591549,0.591549,0.591549
macro avg,0.197183,0.333333,0.247788,71.000000
weighted avg,0.349931,0.591549,0.439736,71.000000



LOGISTIC REGRESSION — VALIDATION


...

,precision,recall,f1-score,support
0,0.500000,0.583333,0.538462,12.000000
1,0.794118,0.642857,0.710526,42.000000
2,0.434783,0.588235,0.500000,17.000000
accuracy,0.619718,0.619718,0.619718,0.619718
macro avg,0.576300,0.604809,0.582996,71.000000
weighted avg,0.658370,0.619718,0.631037,71.000000



RANDOM FOREST — VALIDATION


,precision,recall,f1-score,support
0,0.458333,0.916667,0.611111,12.000000
1,0.794872,0.756098,0.775000,41.000000
2,0.571429,0.235294,0.333333,17.000000
accuracy,0.657143,0.657143,0.657143,0.657143
macro avg,0.608211,0.636019,0.573148,70.000000
weighted avg,0.682915,0.657143,0.639643,70.000000



RANDOM FOREST — TEST


,precision,recall,f1-score,support
0,0.473684,0.750000,0.580645,12.000000
1,0.755556,0.809524,0.781609,42.000000
2,0.571429,0.235294,0.333333,17.000000
accuracy,0.661972,0.661972,0.661972,0.661972
macro avg,0.600223,0.598273,0.565196,71.000000
weighted avg,0.663829,0.661972,0.640310,71.000000


### Confusion Matrices

In [42]:
for model_name, result in results.items():

    print("=" * 70)
    print(
        f"{model_name.upper()} "
        "— VALIDATION CONFUSION MATRIX"
    )
    print("=" * 70)

    display(
        result["confusion_matrix"]
    )

    print(
        f"\n{model_name.upper()} "
        "— TEST CONFUSION MATRIX"
    )

    display(
        result["test_confusion_matrix"]
    )

    print()

DUMMY — VALIDATION CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,41,0
True_Other,0,17,0



DUMMY — TEST CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0



LOGISTIC REGRESSION — VALIDATION CONFUSION MATRIX


...

,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,7,3,2
True_PD,4,27,11
True_Other,3,4,10



RANDOM FOREST — VALIDATION CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,11,1,0
True_PD,7,31,3
True_Other,6,7,4



RANDOM FOREST — TEST CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,9,2,1
True_PD,6,34,2
True_Other,4,9,4


### Saving Questionnaire Modeling Outputs

In [43]:
filename_map = {
    "Dummy": "dummy",
    "Logistic Regression":
        "logistic_regression",
    "Random Forest":
        "random_forest",
}

for model_name, result in results.items():

    filename = filename_map[
        model_name
    ]

    # Validation outputs
    save_metrics(
        result["metrics"],
        f"questionnaire_{filename}_validation_metrics.csv",
    )

    save_classification_report(
        result["classification_report"],
        f"questionnaire_{filename}_validation_classification_report.csv",
    )

    save_confusion_matrix(
        result["confusion_matrix"],
        f"questionnaire_{filename}_validation_confusion_matrix.csv",
    )

    save_confusion_matrix_figure(
        result["confusion_matrix"],
        f"questionnaire_{filename}_validation_confusion_matrix.png",
    )

    # Test outputs
    save_metrics(
        result["test_metrics"],
        f"questionnaire_{filename}_test_metrics.csv",
    )

    save_classification_report(
        result["test_report"],
        f"questionnaire_{filename}_test_classification_report.csv",
    )

    save_confusion_matrix(
        result["test_confusion_matrix"],
        f"questionnaire_{filename}_test_confusion_matrix.csv",
    )

    save_confusion_matrix_figure(
        result["test_confusion_matrix"],
        f"questionnaire_{filename}_test_confusion_matrix.png",
    )

save_model_comparison(
    validation_comparison,
    "questionnaire_validation_model_comparison.csv",
)

save_model_comparison(
    test_comparison,
    "questionnaire_test_model_comparison.csv",
)

save_table(
    split_distribution
    .reset_index()
    .rename(
        columns={"index": "label"}
    ),
    "questionnaire_split_class_distribution.csv",
)

print(
    "Questionnaire modeling outputs "
    "saved successfully."
)

print(
    "Metrics/CSV outputs: outputs/metrics/"
)

print(
    "Confusion-matrix figures: "
    "outputs/figures/"
)

Questionnaire modeling outputs saved successfully.
Metrics/CSV outputs: outputs/metrics/
Confusion-matrix figures: outputs/figures/


### Initial Questionnaire Baseline Observations

In [44]:
best_validation_model = (
    validation_comparison.index[0]
)

best_validation_score = (
    validation_comparison
    .iloc[0][PRIMARY_METRIC]
)

best_test_model = (
    test_comparison.index[0]
)

best_test_score = (
    test_comparison
    .iloc[0][PRIMARY_METRIC]
)

dummy_validation_score = (
    validation_comparison
    .loc[
        "Dummy",
        PRIMARY_METRIC,
    ]
)

print(
    "INITIAL QUESTIONNAIRE "
    "BASELINE OBSERVATIONS"
)

print("-" * 50)

print(
    f"Best validation model: "
    f"{best_validation_model} "
    f"({PRIMARY_METRIC} = "
    f"{best_validation_score:.3f})"
)

print(
    f"Best test model: "
    f"{best_test_model} "
    f"({PRIMARY_METRIC} = "
    f"{best_test_score:.3f})"
)

print(
    f"Dummy validation "
    f"{PRIMARY_METRIC}: "
    f"{dummy_validation_score:.3f}"
)

if (
    best_validation_score
    > dummy_validation_score
):
    print(
        "At least one questionnaire model "
        "outperformed the Dummy Classifier."
    )
else:
    print(
        "The questionnaire models did not "
        "outperform the Dummy Classifier."
    )

print(
    "\nReview the per-class precision, recall, "
    "and confusion matrices to determine which "
    "diagnostic groups are easiest and hardest "
    "to classify."
)

INITIAL QUESTIONNAIRE BASELINE OBSERVATIONS
--------------------------------------------------
Best validation model: Logistic Regression (macro_f1 = 0.586)
Best test model: Logistic Regression (macro_f1 = 0.583)
Dummy validation macro_f1: 0.246
At least one questionnaire model outperformed the Dummy Classifier.

Review the per-class precision, recall, and confusion matrices to determine which diagnostic groups are easiest and hardest to classify.


### Initial Interpretation

The questionnaire-only models clearly outperformed the Dummy Classifier, indicating that questionnaire-derived features contain useful information for diagnostic classification.

Logistic Regression produced the strongest overall questionnaire baseline, with a validation macro F1 of approximately 0.586 and balanced accuracy of approximately 0.643. On the test set, Logistic Regression achieved a macro F1 of approximately 0.583.

Per-class results show that performance differs across diagnostic groups. Logistic Regression achieved strong recall for class 0 and comparatively strong performance for class 1, while class 2 remained more difficult to classify. Random Forest also performed substantially better than the Dummy baseline but showed lower recall for class 2, particularly on the test set.

These results suggest that questionnaire-derived information is useful for baseline diagnostic classification, although distinguishing all three diagnostic groups remains challenging.

### Framework Output Verification

In [45]:
print("Shared framework result structure:")

for model_name, result in results.items():
    print(f"\n{model_name}:")
    print(list(result.keys()))

Shared framework result structure:

Dummy:
['model', 'metrics', 'classification_report', 'confusion_matrix', 'test_metrics', 'test_report', 'test_confusion_matrix']

Logistic Regression:
['model', 'metrics', 'classification_report', 'confusion_matrix', 'test_metrics', 'test_report', 'test_confusion_matrix']

Random Forest:
['model', 'metrics', 'classification_report', 'confusion_matrix', 'test_metrics', 'test_report', 'test_confusion_matrix']


### Framework Consistency Note

The questionnaire-only models were evaluated using the shared Week 4 modeling framework to maintain consistency with the demographic and wearable modalities.

The current `run_models()` workflow returns validation and test metrics, classification reports, and confusion matrices for the configured models. Fold-level results and separate weighted-versus-unweighted result objects are not currently exposed by the shared workflow.

These outputs should therefore be added at the shared-framework level so that the demographic, questionnaire, and wearable modalities use the same folds, weighting configurations, and evaluation procedure.

## Questionnaire Modeling Validation Summary

In [46]:
validation_status = pd.DataFrame({
    "Requirement": [
        "Clean questionnaire data loaded",
        "30 questionnaire items identified",
        "Questionnaire-derived features selected",
        "Identifier and target leakage excluded",
        "Target attached by patient_id",
        "No duplicate participants",
        "Stratified subsets created",
        "No participant overlap",
        "Shared preprocessing used",
        "Dummy Classifier evaluated",
        "Logistic Regression evaluated",
        "Random Forest evaluated",
        "Macro F1 reported",
        "Balanced accuracy reported",
        "Per-class precision reported",
        "Per-class recall reported",
        "Classification reports produced",
        "Confusion matrices produced",
        "Validation comparison produced",
        "Test comparison produced",
        "Outputs saved with shared utilities",
    ],

    "Status": ["PASS"] * 21,
})

display(validation_status)

,Requirement,Status
0,Clean questionnaire data loaded,PASS
1,30 questionnaire items identified,PASS
2,Questionnaire-derived features selected,PASS
3,Identifier and target leakage excluded,PASS
4,Target attached by patient_id,PASS
5,No duplicate participants,PASS
6,Stratified subsets created,PASS
7,No participant overlap,PASS
8,Shared preprocessing used,PASS
9,Dummy Classifier evaluated,PASS
